# M4: Scheduler Loop

M3 rejected a second request with `503` while the engine was busy. M4 queues it instead.

```text
handler thread                          scheduler thread
--------------                          ----------------
Engine.generate()
  scheduler.submit(req) --recv_queue-->  recv_requests()          recv_queue -> waiting_queue
  req.done.wait()   (sleeps)             get_next_batch_to_run()  oldest waiting req, if idle
                                         run_batch()              engine._run(req)
                    <-- done.set() ----- process_batch_result()   free running_req, wake handler
```

The code lives in `scheduler.py`, `engine.py`, and `req.py`. This notebook starts the real server and checks four things:

- Concurrent requests all get `200`, no more `503`
- They run one at a time, in arrival order
- An idle scheduler uses no CPU
- A failing request gets `500`, and the server keeps serving

## Setup

Same as M3: start uvicorn in a background thread so the notebook can act as the client. `Engine()` now also starts the scheduler thread.

Port 30000 is SGLang's default. Stop any `python server.py` you started in a terminal first, or the port is taken.

In [1]:
import json
import threading
import time

import httpx
import uvicorn

from engine import Engine
from server import create_app

engine = Engine()
print(engine.device)

server = uvicorn.Server(uvicorn.Config(create_app(engine), host="127.0.0.1", port=30000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

client = httpx.Client(base_url="http://127.0.0.1:30000", timeout=300)


def show(resp):
    print(resp.status_code, resp.reason_phrase)
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print(repr(resp.text))  # not every error body is JSON

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps


## Record When Each Request Runs

Wrap `engine._run` to log when the model starts and stops on each request. Setting `engine._run` on the instance shadows the class method. `del engine._run` restores it.

The scheduler calls `engine._run` through a lambda, so it picks up the wrapper even though the scheduler was built first.

In [2]:
timeline = []  # (prompt, start, end), seconds since t0
t0 = time.perf_counter()
original_run = type(engine)._run


def timed_run(req):
    start = time.perf_counter() - t0
    original_run(engine, req)
    timeline.append((req.prompt, start, time.perf_counter() - t0))


engine._run = timed_run

## Concurrent Requests Queue Up

Send A, B, and C from three threads, 0.2 s apart, so their arrival order is fixed. In M3, B and C would get `503`. Now all three wait their turn.

In [3]:
prompts = ["A: Write a story about a dragon.", "B: Write a poem about the sea.", "C: Explain what a GPU does."]
results = {}
sent_at = {}


def post(prompt):
    sent_at[prompt] = time.perf_counter() - t0
    results[prompt] = client.post("/generate", json={"text": prompt, "sampling_params": {"max_new_tokens": 48}})


timeline.clear()
t0 = time.perf_counter()
threads = [threading.Thread(target=post, args=(p,)) for p in prompts]
for t in threads:
    t.start()
    time.sleep(0.2)
for t in threads:
    t.join()

for p in prompts:
    print(p[:1], results[p].status_code, results[p].json()["meta_info"]["finish_reason"])

A 200 length
B 200 length
C 200 length


## They Run One at a Time, in Order

- **queued** is how long the request waited in `waiting_queue` before the model started on it.
- **ran** is how long the model ran on it.

Each request starts only after the previous one ends. B and C spend most of their time queued, because each request waits for everything ahead of it to finish. M7 batching attacks this wait.

In [4]:
print(f"{'req':<4}{'sent':>8}{'start':>8}{'end':>8}{'queued':>9}{'ran':>8}")
for prompt, start, end in timeline:
    print(f"{prompt[:1]:<4}{sent_at[prompt]:>8.2f}{start:>8.2f}{end:>8.2f}{start - sent_at[prompt]:>9.2f}{end - start:>8.2f}")

starts_after_prev_end = all(timeline[i][1] >= timeline[i - 1][2] for i in range(1, len(timeline)))
print("order:", [p[:1] for p, _, _ in timeline], "| serial:", starts_after_prev_end)

req     sent   start     end   queued     ran
A       0.00    0.03    2.72     0.03    2.69
B       0.20    2.72    5.27     2.52    2.55
C       0.41    5.27    7.78     4.86    2.51
order: ['A', 'B', 'C'] | serial: True


## Look Inside the Scheduler While A Runs

Send A, B, and C again, and look at the scheduler's state while A is on the model:

- `running_req` is A
- B and C wait in `waiting_queue`, or are still in `recv_queue` if the scheduler has not reached its next `recv_requests()` yet

In M4 it is always the second case. `recv_requests()` only runs between steps, and the current step lasts until A is done. So B and C sit in the thread-safe inbox, and the scheduler moves them to `waiting_queue` in one go once A finishes.

In [5]:
timeline.clear()
t0 = time.perf_counter()
threads = [threading.Thread(target=post, args=(p,)) for p in prompts]
for t in threads:
    t.start()
    time.sleep(0.2)

s = engine.scheduler
print("running_req  :", s.running_req.prompt[:1] if s.running_req else None)
print("waiting_queue:", [r.prompt[:1] for r in s.waiting_queue])
print("recv_queue   :", s.recv_queue.qsize(), "request(s)")

for t in threads:
    t.join()
print("after        :", s.running_req, list(s.waiting_queue), s.recv_queue.qsize())

running_req  : A
waiting_queue: []
recv_queue   : 2 request(s)
after        : None [] 0


## An Idle Scheduler Uses No CPU

When there is no work, `recv_requests()` blocks on `recv_queue.get()` and the OS puts the scheduler thread to sleep. `time.process_time()` counts CPU time across all threads in this process, so a busy loop would show about 1 s here.

In [6]:
start = time.process_time()
time.sleep(1)
print(f"CPU used while idle for 1 s: {time.process_time() - start:.3f}s")

CPU used while idle for 1 s: 0.009s


## A Failing Request Gets 500, and the Server Keeps Serving

Make `_run` raise. `event_loop` catches the error, stores it on `req.error`, and still sets `req.done`. `Engine.generate` re-raises it, the handler does not catch it, and FastAPI turns it into `500 Internal Server Error`.

uvicorn also logs the traceback. Read it bottom-up: `failing_run` ← `run_batch` ← `event_loop` (scheduler thread), then `Engine.generate` ← `generate` handler (handler thread), where the error is re-raised.

Without the `try/except` in `event_loop`, the error would kill the scheduler thread and every later request would hang on `req.done.wait()` forever.

In [7]:
def failing_run(req):
    raise RuntimeError("model crashed")


engine._run = failing_run
# uvicorn drops the connection after an unhandled error; close it so the next request opens a fresh one
show(client.post("/generate", json={"text": "Hello"}, headers={"Connection": "close"}))

del engine._run  # back to the real model
show(client.post("/generate", json={"text": "Hello", "sampling_params": {"max_new_tokens": 8}}))

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/protocols/http/h11_impl.py", line 411, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/middl

500 Internal Server Error
'Internal Server Error'
200 OK
{
  "text": " Question! I'm a little confused about",
  "output_ids": [
    15846,
    0,
    358,
    2776,
    264,
    2632,
    21815,
    911
  ],
  "meta_info": {
    "id": "5fb4aac63dea4dbfa1945b1d984c5c66",
    "finish_reason": "length",
    "prompt_tokens": 1,
    "completion_tokens": 8
  }
}


## Summary

| | M3 | M4 |
|---|---|---|
| Second request while busy | `503`, retry later | waits in the queue, then `200` |
| Who runs the model | handler thread | scheduler thread |
| Order | whoever gets the lock | FIFO |
| Model error | `500` from the handler | `500`; the scheduler thread survives |
| Handoff | none | `recv_queue` → `waiting_queue` |
| Waiting | none | `req.done.wait()` on a `threading.Event` |

One step of the loop still runs one request to the end, so later requests wait for the whole of every request ahead of them. The loop's four-step shape stays. Later milestones change what each step does.

**Compare with SGLang**: `event_loop_normal()` in `managers/scheduler.py`, with the same four steps. SGLang's scheduler runs in its own process and gets requests over ZMQ, not a `queue.Queue`. The HTTP side waits on a per-request `asyncio.Event` in `ReqState` (`managers/tokenizer_manager.py`).

## Shutdown

In [8]:
client.close()
server.should_exit = True